# Calendar Dimension Table

Builds the standard date dimension `adb_caresync.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31, including keys, year/quarter/month/week attributes, ISO week fields, period boundaries and flags.

In [0]:
-- Standard calendar (date dimension): one row per day, 1940-01-01 to 2030-12-31
CREATE OR REPLACE TABLE adb_medilake.silver.calendar
COMMENT 'Standard calendar date dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                 AS date_key,
  d                                                       AS date,
  -- Year
  year(d)                                                 AS year,
  CASE WHEN (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0
       THEN TRUE ELSE FALSE END                           AS is_leap_year,
  make_date(year(d), 1, 1)                                AS first_day_of_year,
  make_date(year(d), 12, 31)                              AS last_day_of_year,
  dayofyear(d)                                            AS day_of_year,
  -- Half year
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END               AS half_year,
  concat('H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END)  AS half_year_name,
  -- Quarter
  quarter(d)                                              AS quarter,
  concat('Q', quarter(d))                                 AS quarter_name,
  concat(year(d), '-Q', quarter(d))                       AS year_quarter,
  CAST(date_trunc('QUARTER', d) AS DATE)                  AS first_day_of_quarter,
  date_sub(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), 1) AS last_day_of_quarter,
  datediff(d, CAST(date_trunc('QUARTER', d) AS DATE)) + 1 AS day_of_quarter,
  -- Month
  month(d)                                                AS month,
  date_format(d, 'MMMM')                                  AS month_name,
  date_format(d, 'MMM')                                   AS month_short_name,
  date_format(d, 'yyyy-MM')                               AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                   AS year_month_key,
  CAST(date_trunc('MONTH', d) AS DATE)                    AS first_day_of_month,
  last_day(d)                                             AS last_day_of_month,
  day(last_day(d))                                        AS days_in_month,
  -- Week
  weekofyear(d)                                           AS week_of_year,          -- ISO 8601 week (Mon-Sun)
  year(date_add(d, 3 - weekday(d)))                       AS iso_year,              -- year of the Thursday in the ISO week
  concat(year(date_add(d, 3 - weekday(d))), '-W', lpad(weekofyear(d), 2, '0')) AS iso_year_week,
  CAST(ceil(day(d) / 7.0) AS INT)                         AS week_of_month,         -- days 1-7 = week 1, etc.
  date_sub(d, weekday(d))                                 AS week_start_date,       -- Monday
  date_add(date_sub(d, weekday(d)), 6)                    AS week_end_date,         -- Sunday
  -- Day
  day(d)                                                  AS day_of_month,
  weekday(d) + 1                                          AS day_of_week,           -- ISO: 1=Monday ... 7=Sunday
  date_format(d, 'EEEE')                                  AS day_name,
  date_format(d, 'EEE')                                   AS day_short_name,
  -- Flags
  weekday(d) >= 5                                         AS is_weekend,
  weekday(d) < 5                                          AS is_weekday,
  day(d) = 1                                              AS is_month_start,
  d = last_day(d)                                         AS is_month_end,
  d = CAST(date_trunc('QUARTER', d) AS DATE)              AS is_quarter_start,
  d = date_sub(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), 1) AS is_quarter_end,
  (month(d) = 1 AND day(d) = 1)                           AS is_year_start,
  (month(d) = 12 AND day(d) = 31)                         AS is_year_end
FROM dates
ORDER BY d;

In [0]:
-- Validation: row count, date range, uniqueness of date_key
SELECT
  count(*)                         AS total_rows,
  count(DISTINCT date_key)         AS distinct_date_keys,
  min(date)                        AS min_date,
  max(date)                        AS max_date,
  datediff(max(date), min(date)) + 1 AS expected_rows,
  sum(CASE WHEN is_leap_year AND month = 2 AND day_of_month = 29 THEN 1 ELSE 0 END) AS leap_days
FROM adb_medilake.silver.calendar;

In [0]:
SELECT * FROM adb_medilake.silver.calendar
WHERE date IN (DATE'1940-01-01', DATE'2020-02-29', DATE'2024-12-30', DATE'2026-11-08', DATE'2030-12-31')
ORDER BY date;